# CrewAI: Multi-Agent Collaboration, Roles & Task Delegation

**Scenario:** Competitor research → insight generation → marketing angle drafting, handled by a 3-agent CrewAI crew instead of one generalist agent.

This notebook covers:
1. Multi-agent design thinking
2. Agents & role-appropriate tools
3. Tasks, dependencies & sequential process
4. Hierarchical delegation (manager agent)
5. Evaluation & cost comparison (sequential vs hierarchical vs single-agent LangGraph)


## Task 1: Multi-Agent Design Thinking

**Business task:** *Research a competitor, summarize findings, and draft a marketing angle.*

### Why this task, and why 3 agents?
The task naturally splits into three non-overlapping skills  gathering facts, interpreting facts, and persuasive writing. Keeping these separate avoids a common generalist failure mode: mixing research with spin (i.e. the model starts "selling" while it's still supposed to be fact-finding), which quietly corrupts the factual grounding of the output.

| Agent | Role | Goal | Backstory |
|---|---|---|---|
| **Researcher** | Market Research Analyst | Gather accurate, current facts about the competitor's products, pricing, and recent marketing activity, using web search. | "You are a meticulous market researcher with 10 years of experience tracking competitors in the SaaS industry. You dig for verifiable facts, not opinions, and always note where information came from." |
| **Analyst** | Business Insights Analyst | Turn raw research into 3–5 structured, prioritized insights (strengths / weaknesses / opportunities) relevant to our positioning. | "You are a sharp strategic analyst who has worked on competitive-intelligence teams. You don't just summarize, you interpret what findings mean for the business." |
| **Copywriter** | Marketing Content Strategist | Turn the analyst's insights into one stakeholder-ready marketing angle: a headline plus 2–3 supporting points. | "You are a creative marketing copywriter known for punchy, benefit-driven messaging that resonates with decision-makers." |

### Why multiple specialized agents might outperform one generalist  and where that isn't true
Splitting the task lets each agent hold a narrow, focused prompt/persona, which improves role-specific reasoning: the researcher stays fact-only, the analyst reasons over clean findings instead of a mixed fact/opinion blob, and the writer optimizes purely for persuasion instead of also worrying about accuracy. This division of labor tends to produce more accurate *and* more polished output than one model juggling all three concerns at once.

That said, it isn't a free win. For a **simple or low-stakes** version of this task, the coordination overhead  extra LLM calls, higher latency, more moving parts to debug  can cost more (in time, tokens, and money) than the quality gain is worth. A single well-prompted generalist agent is often the better choice when the task is short, low-risk, or doesn't need strict separation of concerns.

In [1]:
!pip install crewai crewai-tools python-dotenv -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 8.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.5/195.5 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 833.2/833.2 kB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4

In [2]:
%pip install -U tavily-python

## Setup / API Keys

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

# Load and verify required API keys as variables
# (needed by later cells, e.g. researcher_llm / manager_llm)
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

if not OPENROUTER_API_KEY:
    raise RuntimeError("OPENROUTER_API_KEY is not set.")

if not TAVILY_API_KEY:
    raise RuntimeError("TAVILY_API_KEY is not set.")

print("OPENROUTER_API_KEY loaded:", bool(OPENROUTER_API_KEY))
print("TAVILY_API_KEY loaded:", bool(TAVILY_API_KEY))

from crewai import Agent, Task, Crew, Process, LLM

OPENROUTER_API_KEY loaded: True
TAVILY_API_KEY loaded: True


In [4]:
import crewai
import crewai_tools

print("CrewAI version:", crewai.__version__)
print("CrewAI Tools version:", getattr(crewai_tools, "__version__", "unknown"))

CrewAI version: 1.15.16
CrewAI Tools version: 1.15.16


In [5]:
print([name for name in dir(crewai_tools) if "Tavily" in name])

['TavilyExtractorTool', 'TavilyGetResearchTool', 'TavilyResearchTool', 'TavilySearchTool']


                    CREW
                     │
                     ▼
              ┌─────────────┐
              │  Researcher │
              └──────┬──────┘
                     │
             ┌───────┴────────┐
             │                │
             ▼                ▼
        OpenRouter          Tavily
           LLM             Web Search
             │                │
             └───────┬────────┘
                     │
                     ▼
                  Research
                     │
                     ▼
                ┌─────────┐
                │ Analyst │
                └────┬────┘
                     │
                     ▼
               ┌───────────┐
               │ Copywriter│
               └───────────┘

## Task 2: Build Agents & Assign Tools

Tool access is kept **role-appropriate**, not shared indiscriminately:

| Agent | Tool(s) | Justification |
|---|---|---|
| Researcher | `TavilySearchTool` (web search) | Needs live, current information  the only agent that touches the outside world. |
| Analyst | *none* | Works purely on the researcher's output; giving it search access risks it going off and re-researching instead of synthesizing, and adds unnecessary cost/latency. |
| Copywriter | *none* (optionally `FileReadTool` for brand-voice guidelines) | Pure generation task grounded in the analyst's insights; a brand-guidelines file is the only external input it plausibly needs. |

Each agent also gets its **own LLM config**  the researcher/analyst use a lower-temperature, fact-oriented setting, while the copywriter uses a higher temperature for more creative phrasing.

In [6]:
from pydantic import BaseModel, Field
from typing import Type
from crewai.tools import BaseTool
from tavily import TavilyClient

# TAVILY CLIENT

tavily_client = TavilyClient(
    api_key=TAVILY_API_KEY
)


# TAVILY TOOL INPUT SCHEMA

class TavilySearchInput(BaseModel):

    query: str = Field(
        ...,
        description="The web search query to search for."
    )


# CUSTOM TAVILY SEARCH TOOL

class TavilySearchTool(BaseTool):

    name: str = "tavily_search"

    description: str = """
    Search the web for current and historical information.

    IMPORTANT:
    - For current information, use the current date/context.
    - Never automatically append an old year such as 2023, 2024, or 2025.
    - For recent news, campaigns, announcements, or product activity,
      search using the current date and recent time window.
    - Prefer official sources when the task asks for official information.
    - Return source URLs and publication dates whenever available.
    """

    args_schema: Type[BaseModel] = TavilySearchInput

    def _run(self, query: str) -> str:

        response = tavily_client.search(
            query=query,
            max_results=5,
            search_depth="advanced"
        )

        results = response.get("results", [])

        if not results:
            return "No search results found."

        formatted_results = []

        for i, result in enumerate(results, start=1):

            title = result.get("title", "No title")
            url = result.get("url", "No URL")
            content = result.get("content", "No content")
            published_date = result.get(
                "published_date",
                "Publication date not available"
            )

            formatted_results.append(
                f"""
RESULT {i}

Title: {title}

URL: {url}

Publication Date: {published_date}

Content:
{content}
"""
            )

        return "\n".join(formatted_results)


# CREATE SEARCH TOOL

search_tool = TavilySearchTool()


# PER-AGENT LLM CONFIGURATION

researcher_llm = LLM(
    model="openrouter/openai/gpt-4o-mini",
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
    temperature=0.2,
)

analyst_llm = LLM(
    model="openrouter/openai/gpt-4o-mini",
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
    temperature=0.3,
)

writer_llm = LLM(
    model="openrouter/openai/gpt-4o-mini",
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
    temperature=0.7,
)


# RESEARCHER AGENT

researcher = Agent(

    role="Market Research Analyst",

    goal=(
        "Gather accurate, current, source-backed information about "
        "{competitor}'s products, pricing, and recent marketing activity. "
        "Use web search, prefer official sources, verify publication dates, "
        "and reject stale or unsupported information."
    ),

    backstory=(
        "You are a meticulous market researcher with 10 years of "
        "experience tracking SaaS competitors. You focus on verifiable "
        "facts rather than opinions. You verify dates, distinguish current "
        "information from historical information, and prefer primary "
        "sources such as official pricing pages, product pages, blogs, "
        "newsrooms, and official announcements. "
        "Never invent dates, prices, products, campaigns, or claims."
    ),

    tools=[search_tool],

    llm=researcher_llm,

    verbose=True,

    allow_delegation=False,

    max_iter=8,
)


# ANALYST AGENT

analyst = Agent(

    role="Business Insights Analyst",

    goal=(
        "Distill the research findings into 3-5 evidence-based, "
        "prioritized insights relevant to our company's positioning "
        "against {competitor}. Use only facts supported by the research."
    ),

    backstory=(
        "You are a strategic competitive-intelligence analyst. "
        "You identify business implications from verified evidence, "
        "but you never invent facts or assumptions. "
        "You distinguish clearly between demonstrated strengths, "
        "demonstrated weaknesses, and potential opportunities."
    ),

    tools=[],

    llm=analyst_llm,

    verbose=True,

    allow_delegation=False,

    max_iter=4,
)


# COPYWRITER AGENT

copywriter = Agent(

    role="Marketing Content Strategist",

    goal=(
        "Turn the analyst's validated insights into one concise, "
        "stakeholder-ready marketing angle. Use only information "
        "contained in the analyst's insights and do not invent "
        "capabilities, pricing, features, or advantages for our company."
    ),

    backstory=(
        "You are a careful B2B marketing strategist known for concise, "
        "evidence-based positioning. You can recommend a positioning "
        "direction, but you must never present an unverified capability "
        "of our company as an existing fact."
    ),

    tools=[],

    llm=writer_llm,

    verbose=True,

    allow_delegation=False,

    max_iter=3,
)


print("✅ All agents and tools configured successfully.")

✅ All agents and tools configured successfully.


## Task 3: Define Tasks & Process (Sequential)

Each `Task` has a clear `description`, `expected_output`, and  for the analyst/copywriter  a `context` dependency pointing at the earlier task(s) whose output they consume.

In [7]:
!pip install -q nest_asyncio

In [8]:
# TASK 3 — CREW WORKFLOW

from crewai import Task, Crew, Process

from datetime import datetime, timedelta

import asyncio
import nest_asyncio


# DATE WINDOW

TODAY = datetime.now().date()

SIX_MONTHS_AGO = TODAY - timedelta(days=180)

TODAY_STR = TODAY.isoformat()

CUTOFF_STR = SIX_MONTHS_AGO.isoformat()


print("Current date:", TODAY_STR)
print("Recent activity cutoff:", CUTOFF_STR)


# COMPETITOR

competitor = "Notion"


# RESEARCH TASK

research_task = Task(

    description=f"""
Research {competitor}'s products, current pricing,
and recent marketing/announcement activity.

CURRENT DATE:
{TODAY_STR}

RECENT-ACTIVITY CUTOFF:
{CUTOFF_STR}

============================================================
RESEARCH SCOPE
============================================================

A. CURRENT PRODUCTS

Research {competitor}'s currently offered
products/platform offerings.

B. CURRENT PRICING

Research {competitor}'s current pricing tiers/plans.

C. RECENT MARKETING CAMPAIGNS

Find marketing campaigns published between
{CUTOFF_STR} and {TODAY_STR}.

D. RECENT ANNOUNCEMENTS / PRODUCT ACTIVITY

Find product launches, major announcements,
updates, or other official activity published
between {CUTOFF_STR} and {TODAY_STR}.

============================================================
SEARCH RULES
============================================================

1. Use the Tavily web search tool.

2. Do not automatically search using old hard-coded years.

3. Use terms such as:

   - current
   - latest
   - official
   - pricing
   - plans
   - announcements
   - newsroom
   - product updates
   - blog
   - campaign

4. Prefer official {competitor} sources.

5. For pricing, prioritize the official pricing page.

6. For products, prioritize official product pages.

7. For announcements, prioritize official blog/newsroom pages.

8. For campaigns, prioritize official marketing/blog/newsroom
   sources.

9. Third-party sources may only be used when official sources
   are unavailable or useful for factual confirmation.

10. Never use an old article as proof of current information.

============================================================
CURRENT INFORMATION VALIDATION
============================================================

For CURRENT products and pricing:

- Verify that the source represents the current offering.
- Prefer live official sources.
- Do not rely on old articles.
- If sources conflict, prefer the newer official source.
- Never guess prices.

If current pricing cannot be verified, state:

"Current pricing could not be verified from the available sources."

============================================================
RECENT ACTIVITY VALIDATION
============================================================

For every marketing campaign or announcement:

1. Identify the original publication date.

2. Confirm:

   {CUTOFF_STR} <= publication date <= {TODAY_STR}

3. If the date is outside this window,
   do not classify it as recent.

4. If the publication date cannot be verified,
   do not classify it as recent.

5. Do not confuse:

   - publication date
   - update date
   - crawl date
   - search result date
   - video upload date
   - campaign launch date

============================================================
ANTI-HALLUCINATION RULES
============================================================

- Do not invent dates.
- Do not invent prices.
- Do not invent products.
- Do not invent campaigns.
- Do not infer campaign dates.
- Do not convert historical facts into current facts.
- Do not add opinions.
- Do not add recommendations.
- Do not infer business implications.
- Do not label something as a strength or weakness.
- Only report what sources directly support.

============================================================
OUTPUT
============================================================

Return 6-10 factual findings.

Each finding MUST follow:

Finding 1: Fact: <one factual statement> | Date: <date> | Source: <source name and URL>

Finding 2: Fact: <one factual statement> | Date: <date> | Source: <source name and URL>

Continue sequentially.

For current products/pricing:

Date may be:

Current as verified on {TODAY_STR}

For recent campaigns/announcements:

Date MUST contain the verified publication date.

That date MUST fall between:

{CUTOFF_STR}

and

{TODAY_STR}

Cover at least:

- current products
- current pricing
- recent marketing activity
- recent announcements/product activity

Do not include unsupported information.
""",

    expected_output=f"""
A numbered list containing 6-10 factual findings.

Every finding must use:

Finding N: Fact: <one factual statement> | Date: <date> | Source: <source name and URL>

Requirements:

- Current pricing/products must be verified as current.
- Recent campaigns/announcements must have publication dates.
- Recent activity dates must fall between {CUTOFF_STR}
  and {TODAY_STR}.
- No invented dates.
- No invented prices.
- No unsupported claims.
- No recommendations.
- No opinions.
""",

    agent=researcher,
)


# ANALYSIS TASK

analysis_task = Task(

    description="""
Review ONLY the research findings produced by the
previous task.

Do NOT perform additional web research.

Identify 3-5 prioritized business insights that are
directly supported by the research findings.

============================================================
EVIDENCE RULES
============================================================

Use ONLY information contained in the research findings.

Do not introduce:

- new competitor facts
- new prices
- new products
- new campaigns
- new dates
- external knowledge
- assumptions
- unsupported interpretations

============================================================
STRENGTH / WEAKNESS / OPPORTUNITY
============================================================

Strength:
Only when the research explicitly demonstrates
a competitor advantage.

Weakness:
Only when the research explicitly demonstrates
a competitor limitation, disadvantage, constraint,
or negative condition.

Opportunity:
Only when the research provides evidence of a
market or positioning opportunity.

Do not label something a weakness merely because:

- the competitor has a high price
- the competitor uses a particular channel
- the competitor has many features
- the competitor spends money on marketing

Those facts alone do not prove a weakness.

============================================================
NUMBERING
============================================================

Research finding numbers and insight numbers
are different.

Example:

Insight 1 ... | Source Finding: 4

The Source Finding number MUST correspond to
an actual research finding.

============================================================
OUTPUT
============================================================

Return exactly 3-5 insights.

Each insight must be exactly ONE sentence.

Use:

Insight 1: [Strength/Weakness/Opportunity] <one sentence> | Source Finding: <number>

Insight 2: [Strength/Weakness/Opportunity] <one sentence> | Source Finding: <number>

Continue sequentially.

Do not create unsupported insights.
""",

    expected_output="""
A numbered list containing 3-5 evidence-based insights.

Each insight must:

- contain exactly one sentence
- use Strength, Weakness, or Opportunity
- reference an existing research finding
- be fully supported by the research
- contain no new facts
- contain no unsupported assumptions
""",

    agent=analyst,

    context=[research_task],
)


# MARKETING TASK

marketing_task = Task(

    description="""
Review ONLY the analyst's insights.

Create one concise stakeholder-ready marketing
positioning angle for our company.

============================================================
IMPORTANT
============================================================

You ONLY know what is contained in the analyst's insights.

DO NOT invent:

- our company's prices
- our company's features
- our company's products
- our company's capabilities
- our company's customer base
- our company's support model
- our company's advantages
- results or claims about our company

You may phrase a positioning DIRECTION,
but do not state an unverified capability
as an existing fact.

Example:

Instead of:

"Our affordable pricing beats Notion."

Use:

"Position the offering around price competitiveness."

============================================================
OUTPUT
============================================================

Use exactly:

Headline: <fewer than 12 words>

Bullet 1: <supporting marketing point> | Insight: <insight number>

Bullet 2: <supporting marketing point> | Insight: <insight number>

Bullet 3: <supporting marketing point> | Insight: <insight number>

Provide 2-3 supporting bullets.

Every bullet must reference an existing insight number.

Do NOT use research finding numbers as insight numbers.
""",

    expected_output="""
A short stakeholder-ready marketing brief.

Exactly:

Headline: <fewer than 12 words>

Bullet 1: <supporting marketing point> | Insight: <existing insight number>

Bullet 2: <supporting marketing point> | Insight: <existing insight number>

Bullet 3: <supporting marketing point> | Insight: <existing insight number>

Do not invent company capabilities.
Do not introduce new competitor facts.
Do not use research finding numbers as insight numbers.
""",

    agent=copywriter,

    context=[analysis_task],
)


# SEQUENTIAL CREW

sequential_crew = Crew(

    agents=[
        researcher,
        analyst,
        copywriter
    ],

    tasks=[
        research_task,
        analysis_task,
        marketing_task
    ],

    process=Process.sequential,

    verbose=True,
)


# RUN CREW

nest_asyncio.apply()


async def run_sequential_crew():

    result = await sequential_crew.kickoff_async(
        inputs={
            "competitor": competitor
        }
    )

    return result


# EXECUTE

result = asyncio.get_event_loop().run_until_complete(
    run_sequential_crew()
)


# FINAL OUTPUT

print("\n")
print("=" * 70)
print("FINAL CREW OUTPUT")
print("=" * 70)
print(result)

Current date: 2026-08-14
Recent activity cutoff: 2026-02-15


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 09c83cc9-1f22-4595-af25-f76d37bd9ce3                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│  Research Notion's products, current pricing,                                                                   │
│  and recent marketing/announcement activity.                                                                    │
│                                                                                                                 │
│  CURRENT DATE:                                                                                                  │
│  2026-08-14                                                                                                     │
│                                                                                                                 │
│  RECENT-ACTIVITY CUTOFF:                                                                                        │
│  2026-02-15                                                                                                     │
│                                                                                                                 │
│  ============================================================                                                   │
│  RESEARCH SCOPE                                                                                                 │
│  ============================================================                                                   │
│                                                                                                                 │
│  A. CURRENT PRODUCTS                                                                                            │
│                                                                                                                 │
│  Research Notion's currently offered                                                                            │
│  products/platform offerings.                                                                                   │
│                                                                                                                 │
│  B. CURRENT PRICING                                                                                             │
│                                                                                                                 │
│  Research Notion's current pricing tiers/plans.                                                                 │
│                                                                                                                 │
│  C. RECENT MARKETING CAMPAIGNS                                                                                  │
│                                                                                                                 │
│  Find marketing campaigns published between                                                                     │
│  2026-02-15 and 2026-08-14.                                                                                     │
│                                                                                                                 │
│  D. RECENT ANNOUNCEMENTS / PRODUCT ACTIVITY                                                                     │
│                                                                                                                 │
│  Find product launches, major announcements,                                                                    │
│  updates, or other official activity published         

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Market Research Analyst                                                                                 │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Research Notion's products, current pricing,                                                                   │
│  and recent marketing/announcement activity.                                                                    │
│                                                                                                                 │
│  CURRENT DATE:                                                                                                  │
│  2026-08-14                                                                                                     │
│                                                                                                                 │
│  RECENT-ACTIVITY CUTOFF:                                                                                        │
│  2026-02-15                                                                                                     │
│                                                                                                                 │
│  ============================================================                                                   │
│  RESEARCH SCOPE                                                                                                 │
│  ============================================================                                                   │
│                                                                                                                 │
│  A. CURRENT PRODUCTS                                                                                            │
│                                                                                                                 │
│  Research Notion's currently offered                                                                            │
│  products/platform offerings.                                                                                   │
│                                                                                                                 │
│  B. CURRENT PRICING                                                                                             │
│                                                                                                                 │
│  Research Notion's current pricing tiers/plans.                                                                 │
│                                                                                                                 │
│  C. RECENT MARKETING CAMPAIGNS                                                                                  │
│                                                                                                                 │
│  Find marketing campaigns published between                                                                     │
│  2026-02-15 and 2026-08-14.                                                                                     │
│                                                                                                                 │
│  D. RECENT ANNOUNCEMENTS / PRODUCT ACTIVITY                                                                     │
│                                                                                                                 │
│  Find product launches, major announcements,           

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: tavily_search                                                                                            │
│  Args: {'query': 'Notion current products site:notion.so'}                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Marketing Content Strategist                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Headline: Automate Workflows with Enhanced Control and Visibility                                              │
│                                                                                                                 │
│  Bullet 1: Highlight opportunities for workflow automation and task management | Insight: 1                     │
│                                                                                                                 │
│  Bullet 2: Emphasize productivity enhancements from accessible AI features | Insight: 2                         │
│                                                                                                                 │
│  Bullet 3: Position around data residency options appealing to enterprises | Insight: 3                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Review ONLY the analyst's insights.                                                                            │
│                                                                                                                 │
│  Create one concise stakeholder-ready marketing                                                                 │
│  positioning angle for our company.                                                                             │
│                                                                                                                 │
│  ============================================================                                                   │
│  IMPORTANT                                                                                                      │
│  ============================================================                                                   │
│                                                                                                                 │
│  You ONLY know what is contained in the analyst's insights.                                                     │
│                                                                                                                 │
│  DO NOT invent:                                                                                                 │
│                                                                                                                 │
│  - our company's prices                                                                                         │
│  - our company's features                                                                                       │
│  - our company's products                                                                                       │
│  - our company's capabilities                                                                                   │
│  - our company's customer base                                                                                  │
│  - our company's support model                                                                                  │
│  - our company's advantages                                                                                     │
│  - results or claims about our company                                                                          │
│                                                                                                                 │
│  You may phrase a positioning DIRECTION,                                                                        │
│  but do not state an unverified capability                                                                      │
│  as an existing fact.                                                                                           │
│                                                                                                                 │
│  Example:                                                                                                       │
│                                                                                                                 │
│  Instead of:                                                                                                    │
│                                                                                                                 │
│  "Our affordable pricing beats Notion."                

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



FINAL CREW OUTPUT
Headline: Automate Workflows with Enhanced Control and Visibility

Bullet 1: Highlight opportunities for workflow automation and task management | Insight: 1

Bullet 2: Emphasize productivity enhancements from accessible AI features | Insight: 2

Bullet 3: Position around data residency options appealing to enterprises | Insight: 3


In [9]:
print(research_task.output)
print(analysis_task.output)
print(marketing_task.output)

Finding 1: Fact: Notion currently offers a variety of products including Notion AI, Notion Calendar, and Custom Agents, which help users manage tasks and workflows. | Date: Current as verified on 2026-08-14 | Source: Notion Pricing Plans [Notion](https://www.notion.so/pricing)

Finding 2: Fact: Notion's current pricing includes a Free Plan, Plus Plan at $4/month (annual billing), Business Plan at $10/month (annual billing), and Enterprise Plan with custom pricing. | Date: Current as verified on 2026-08-14 | Source: Notion Pricing Plans [Notion](https://www.notion.so/pricing)

Finding 3: Fact: Notion launched its first international brand campaign titled "For your life’s work," which includes advertisements across various media including billboards in major cities. | Date: June 1, 2026 | Source: Notion Blog [Notion](https://www.notion.so/blog/for-your-lifes-work-notions-first-brand-campaign)

Finding 4: Fact: Notion introduced Custom Agents, which allow users to automate workflows and m

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 09c83cc9-1f22-4595-af25-f76d37bd9ce3                                                                       │
│  Final Output: Headline: Automate Workflows with Enhanced Control and Visibility                                │
│                                                                                                                 │
│  Bullet 1: Highlight opportunities for workflow automation and task management | Insight: 1                     │
│                                                                                                                 │
│  Bullet 2: Emphasize productivity enhancements from accessible AI features | Insight: 2                         │
│                                                                                                                 │
│  Bullet 3: Position around data residency options appealing to enterprises | Insight: 3                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### 1. CrewAI Task Definitions and Context Dependencies

Based on the execution logs provided, the workflow consists of three distinct tasks, chained together using context dependencies so that later tasks build directly upon the outputs of earlier ones.

* **Task 1: Market Research (Agent: Market Research Analyst)**
* **Description:** Research Notion's current products, pricing, and recent marketing/announcements between a strict date window (February 15, 2026, to August 14, 2026). The agent is required to use the Tavily web search tool and prioritize official sources.
* **Expected Output:** A structured list of 6-10 factual findings. Each finding must follow a strict format: `Finding N: Fact: <statement> | Date: <date> | Source: <URL>`.
* **Dependencies:** None. This is the foundational task.


* **Task 2: Business Analysis (Agent: Business Insights Analyst)**
* **Description:** Review *only* the research findings produced by Task 1 without conducting any additional web search. The agent must categorize the findings into actionable business insights (Strength, Weakness, or Opportunity) based purely on the provided evidence.
* **Expected Output:** A numbered list of 3-5 insights, where each insight is exactly one sentence and maps to a specific source finding (e.g., `Insight 1: [Strength/Weakness/Opportunity] <sentence> | Source Finding: <number>`).
* **Dependencies:** Contextually dependent on the output of **Task 1**.


* **Task 3: Marketing Strategy (Agent: Marketing Content Strategist)**
* **Description:** Review *only* the analyst's insights from Task 2 to create a concise, stakeholder-ready marketing positioning angle. The agent is explicitly forbidden from inventing company prices, features, or unverified capabilities.
* **Expected Output:** A headline (under 12 words) and 2-3 supporting bullets. Each bullet must reference an existing insight number (e.g., `Bullet 1: <point> | Insight: <insight number>`).
* **Dependencies:** Contextually dependent on the output of **Task 2**.





### 2. Crew Assembly and Sequential Execution

According to the captured execution logs, the Crew was assembled using a
**Sequential Process** (`Process.sequential`).

* **Execution Flow:** The `Market Research Analyst` started first and
  used the Tavily web search tool to gather information about Notion's
  products, pricing, and recent announcements.

* After Task 1 completed, the output was passed as context to the
  `Business Insights Analyst`, which processed the research findings into
  actionable business insights.

* Finally, the output of Task 2 was passed to the
  `Marketing Content Strategist`, which used the insights to produce the
  final marketing positioning output.

The sequential hand-off followed the intended workflow, with each agent
processing the output from the previous stage. However, the output format
required explicit guardrails to prevent confusion between Research
Finding numbers and Insight numbers. After adding these constraints, the
handoff became more predictable and easier to validate.




### 3. Formatting Issue and Prompt Fix

During the sequential workflow, the main output-handoff issue was the
potential confusion between **Research Finding numbers** generated by
Task 1 and **Insight numbers** generated by Task 2.

Task 2 produces business insights that reference the original research
findings. Task 3, however, must reference the **Insight numbers**, not
the original Research Finding numbers. Without an explicit output
contract, the Marketing Content Strategist could incorrectly use a
Research Finding number as an Insight reference, which would break the
data lineage between the analysis and marketing output.

### Prompt / Expected Output Fix

The prompts and expected outputs were made more explicit to create a
clear handoff contract between Task 2 and Task 3.

#### Task 2 Guardrail

The Business Insights Analyst was instructed to distinguish clearly
between research findings and business insights:

> "Research finding numbers and insight numbers are different. The
> Source Finding number MUST correspond to an actual research finding."

The expected output was also structured so that every insight contained
its own identifier and its source finding:

```text
Insight 1: [Strength/Weakness/Opportunity] <sentence>
| Source Finding: <number>

Insight 2: [Strength/Weakness/Opportunity] <sentence>
| Source Finding: <number>

Insight 3: [Strength/Weakness/Opportunity] <sentence>
| Source Finding: <number>
```
#### Task 3 Guardrail

The Marketing Content Strategist was explicitly instructed to use only
the insight identifiers produced by Task 2:

> "Every bullet must reference an existing insight number. Do NOT use
> research finding numbers as insight numbers."

The expected output was defined as:

```text
Headline: <headline under 12 words>

Bullet 1: <marketing point> | Insight: <insight number>
Bullet 2: <marketing point> | Insight: <insight number>
Bullet 3: <marketing point> | Insight: <insight number>

## Task 4: Hierarchical Delegation

Same 3 specialist agents, but now a **manager agent** sits above them, delegates sub-tasks, and reviews/revises their work before finalizing the output. In `Process.hierarchical`, CrewAI auto-manages delegation, so we don't hand-write the manager's own `Task` objects, we supply a `manager_llm` (or a custom `manager_agent`) and let it plan.

In [10]:
from crewai import Crew, Process, LLM

In [32]:
# ============================================================
#  HIERARCHICAL DELEGATION
# MANAGER LLM
# ============================================================

manager_llm = LLM(
    model="openrouter/openai/gpt-4o-mini",
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
    temperature=0.2,
)

print("Manager LLM configured successfully.")

Manager LLM configured successfully.


In [33]:
# ============================================================
# HIERARCHICAL TASKS
# ============================================================

hier_research_task = Task(
    description=f"""
    Research the competitor {{competitor}}.

    CURRENT DATE:
    {TODAY_STR}

    RECENT-ACTIVITY CUTOFF:
    {CUTOFF_STR}

    Investigate the following areas:

    1. Main products and features
    2. Pricing or plans
    3. Recent marketing activity
    4. Recent product announcements or updates

    Gather factual, recent and verifiable information.

    Use the available research/search tool when appropriate.

    Do NOT:
    - create marketing recommendations
    - invent facts
    - speculate beyond the available evidence
    - automatically search using old hard-coded years
    - use an old article as proof of current information

    If sources conflict, prefer the newer official source.
    Only classify a campaign/announcement as "recent" if its
    publication date falls between {CUTOFF_STR} and {TODAY_STR}.

    Your work will be reviewed by a manager and used by another
    specialist for strategic analysis.
    """,

    expected_output="""
    A structured research report containing 6-10 factual findings.

    Use exactly this structure:

    Finding 1:
    Fact: <factual statement>
    Date: <date or "Not specified">
    Source: <source name>
    URL: <source URL>

    Finding 2:
    Fact: <factual statement>
    Date: <date or "Not specified">
    Source: <source name>
    URL: <source URL>

    Continue until 6-10 findings are provided.

    Every finding must be traceable to a source.
    """,

    # IMPORTANT:
    # No context dependency here.
    # The hierarchical manager controls delegation.
)


hier_analysis_task = Task(
    description="""
    Analyze the competitor research produced by the research specialist.

    Identify 3-5 strategic insights.

    Classify every insight as one of:

    - Strength
    - Weakness
    - Opportunity

    Base the analysis ONLY on the research findings provided by
    the research specialist.

    Each insight must explain why the finding matters from a
    business or competitive perspective.

    Do not invent facts or unsupported claims.

    Your output will be reviewed by the manager and passed to
    the marketing specialist.
    """,

    expected_output="""
    A structured list of 3-5 strategic insights.

    Use exactly this format:

    Insight 1:
    Type: Strength / Weakness / Opportunity
    Insight: <one-sentence explanation>
    Source Finding: <finding number(s)>

    Insight 2:
    Type: Strength / Weakness / Opportunity
    Insight: <one-sentence explanation>
    Source Finding: <finding number(s)>

    Continue until 3-5 insights are provided.

    Every insight must reference one or more research findings.
    """
)


hier_marketing_task = Task(
    description="""
    Create a concise stakeholder-ready marketing brief using the
    research and strategic insights produced by the other specialists.

    Produce:

    1. One concise marketing headline
    2. Three supporting marketing points

    The marketing points should respond to the competitor's
    identified strengths, weaknesses or opportunities.

    Do not invent competitor facts.

    Every marketing point must be traceable to an analyst insight.

    Keep the writing professional, concise and persuasive.
    """,

    expected_output="""
    Use exactly this format:

    Headline: <headline under 12 words>

    Bullet 1: <marketing point>
    Insight Reference: <Insight number>

    Bullet 2: <marketing point>
    Insight Reference: <Insight number>

    Bullet 3: <marketing point>
    Insight Reference: <Insight number>

    All three bullets must reference existing analyst insights.
    """
)

print("Hierarchical tasks created successfully.")

Hierarchical tasks created successfully.


In [34]:
# ============================================================
# BUILD HIERARCHICAL CREW
# ============================================================

hierarchical_crew = Crew(
    agents=[
        researcher,
        analyst,
        copywriter
    ],

    tasks=[
        hier_research_task,
        hier_analysis_task,
        hier_marketing_task
    ],

    process=Process.hierarchical,

    # CrewAI uses this LLM for the hierarchical manager.
    manager_llm=manager_llm,

    verbose=True,
)

print("Hierarchical crew created successfully.")

Hierarchical crew created successfully.


                 ┌──────────────────┐
                 │  CrewAI Manager  │
                 │   manager_llm    │
                 └────────┬─────────┘
                          │
             Delegates + Reviews
                          │
        ┌─────────────────┼─────────────────┐
        ↓                 ↓                 ↓
   Researcher          Analyst          Copywriter
        │                 │                 │
    Research          Analysis          Marketing

In [35]:
# ============================================================
# EXECUTE HIERARCHICAL CREW
# ============================================================

import time

start_time = time.perf_counter()

try:
    result_hier = await hierarchical_crew.kickoff_async(
        inputs={
            "competitor": "Notion"
        }
    )

    elapsed_hier = time.perf_counter() - start_time

    print("\n")
    print("=" * 70)
    print("HIERARCHICAL CREW OUTPUT")
    print("=" * 70)
    print(result_hier)

    print("\n")
    print("=" * 70)
    print("HIERARCHICAL EXECUTION TIME")
    print("=" * 70)
    print(f"{elapsed_hier:.2f} seconds")

except Exception as e:

    elapsed_hier = time.perf_counter() - start_time

    print("\n")
    print("=" * 70)
    print("HIERARCHICAL CREW FAILED")
    print("=" * 70)
    print(f"Error: {type(e).__name__}: {e}")
    print(f"Elapsed time: {elapsed_hier:.2f} seconds")

    raise

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 420e8b43-682b-4845-8cf2-75caff228cd3                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│      Research the competitor Notion.                                                                            │
│                                                                                                                 │
│      CURRENT DATE:                                                                                              │
│      2026-08-14                                                                                                 │
│                                                                                                                 │
│      RECENT-ACTIVITY CUTOFF:                                                                                    │
│      2026-02-15                                                                                                 │
│                                                                                                                 │
│      Investigate the following areas:                                                                           │
│                                                                                                                 │
│      1. Main products and features                                                                              │
│      2. Pricing or plans                                                                                        │
│      3. Recent marketing activity                                                                               │
│      4. Recent product announcements or updates                                                                 │
│                                                                                                                 │
│      Gather factual, recent and verifiable information.                                                         │
│                                                                                                                 │
│      Use the available research/search tool when appropriate.                                                   │
│                                                                                                                 │
│      Do NOT:                                                                                                    │
│      - create marketing recommendations                                                                         │
│      - invent facts                                                                                             │
│      - speculate beyond the available evidence                                                                  │
│      - automatically search using old hard-coded years                                                          │
│      - use an old article as proof of current information                                                       │
│                                                                                                                 │
│      If sources conflict, prefer the newer official source.                                                     │
│      Only classify a campaign/announcement as "recent" if its                                                   │
│      publication date falls between 2026-02-15 and 2026-08-14.                                                  │
│                                                                                                                 │
│      Your work will be reviewed by a manager and used b

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Research the competitor Notion.                                                                            │
│                                                                                                                 │
│      CURRENT DATE:                                                                                              │
│      2026-08-14                                                                                                 │
│                                                                                                                 │
│      RECENT-ACTIVITY CUTOFF:                                                                                    │
│      2026-02-15                                                                                                 │
│                                                                                                                 │
│      Investigate the following areas:                                                                           │
│                                                                                                                 │
│      1. Main products and features                                                                              │
│      2. Pricing or plans                                                                                        │
│      3. Recent marketing activity                                                                               │
│      4. Recent product announcements or updates                                                                 │
│                                                                                                                 │
│      Gather factual, recent and verifiable information.                                                         │
│                                                                                                                 │
│      Use the available research/search tool when appropriate.                                                   │
│                                                                                                                 │
│      Do NOT:                                                                                                    │
│      - create marketing recommendations                                                                         │
│      - invent facts                                                                                             │
│      - speculate beyond the available evidence                                                                  │
│      - automatically search using old hard-coded years                                                          │
│      - use an old article as proof of current information                                                       │
│                                                                                                                 │
│      If sources conflict, prefer the newer official source.                                                     │
│      Only classify a campaign/announcement as "recent" if its                                                   │
│      publication date falls between 2026-02-15 and 2026-08-14.                                                  │
│                                                        

╭──────────────────────────────────────── 🔧 Tool Execution Started (#7) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Args: {'task': 'Research the competitor Notion, focusing on their main products and features, pricing or       │
│  plans, recent marketing activity, and recent product announcements or updates. Gather factual, recent...       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Market Research Analyst                                                                                 │
│                                                                                                                 │
│  Task: Research the competitor Notion, focusing on their main products and features, pricing or plans, recent   │
│  marketing activity, and recent product announcements or updates. Gather factual, recent, and verifiable        │
│  information, ensuring that all findings are structured according to the specified format. The research should  │
│  only include information published between 2026-02-15 and 2026-08-14, and all sources must be credible and     │
│  traceable.                                                                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────── ✅ Tool Execution Completed (#9) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Output: **Headline:** Elevate Your Team's Productivity with Tailored Solutions                                 │
│                                                                                                                 │
│  - **Compelling Pricing Strategy:** Notion's flexible pricing tiers cater to diverse business needs, making it  │
│  accessible for teams of all sizes, which presents an opportunity for us to position our offerings as a         │
│  cost-effective alternative for enterprises seeking tailored solutions without compromising on features.        │
│                                                                                                                 │
│  - **Enhanced User Experience:** Recent user experience enhancements in Notion highlight their commitment to    │
│  intuitive design, reinforcing the importance of ease of use; we can leverage this insight by emphasizing our   │
│  advanced user-friendly interfaces that prioritize seamless integration and user satisfaction.                  │
│                                                                                                                 │
│  - **Advanced Automation Features:** Notion's focus on automation capabilities is a strength, but there's an    │
│  opportunity for us to differentiate by promoting our unique data visualization tools that not only automate    │
│  processes but also provide actionable insights, empowering teams to make data-driven decisions faster.         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Headline:** Elevate Your Team's Productivity with Tailored Solutions                                         │
│                                                                                                                 │
│  - **Compelling Pricing Strategy:** Notion's flexible pricing tiers cater to diverse business needs, making it  │
│  accessible for teams of all sizes, which presents an opportunity for us to position our offerings as a         │
│  cost-effective alternative for enterprises seeking tailored solutions without compromising on features.        │
│    Insight Reference: Insight 1                                                                                 │
│                                                                                                                 │
│  - **Enhanced User Experience:** Recent user experience enhancements in Notion highlight their commitment to    │
│  intuitive design, reinforcing the importance of ease of use; we can leverage this insight by emphasizing our   │
│  advanced user-friendly interfaces that prioritize seamless integration and user satisfaction.                  │
│    Insight Reference: Insight 4                                                                                 │
│                                                                                                                 │
│  - **Advanced Automation Features:** Notion's focus on automation capabilities is a strength, but there's an    │
│  opportunity for us to differentiate by promoting our unique data visualization tools that not only automate    │
│  processes but also provide actionable insights, empowering teams to make data-driven decisions faster.         │
│    Insight Reference: Insight 5                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│      Create a concise stakeholder-ready marketing brief using the                                               │
│      research and strategic insights produced by the other specialists.                                         │
│                                                                                                                 │
│      Produce:                                                                                                   │
│                                                                                                                 │
│      1. One concise marketing headline                                                                          │
│      2. Three supporting marketing points                                                                       │
│                                                                                                                 │
│      The marketing points should respond to the competitor's                                                    │
│      identified strengths, weaknesses or opportunities.                                                         │
│                                                                                                                 │
│      Do not invent competitor facts.                                                                            │
│                                                                                                                 │
│      Every marketing point must be traceable to an analyst insight.                                             │
│                                                                                                                 │
│      Keep the writing professional, concise and persuasive.                                                     │
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



HIERARCHICAL CREW OUTPUT
**Headline:** Elevate Your Team's Productivity with Tailored Solutions  

- **Compelling Pricing Strategy:** Notion's flexible pricing tiers cater to diverse business needs, making it accessible for teams of all sizes, which presents an opportunity for us to position our offerings as a cost-effective alternative for enterprises seeking tailored solutions without compromising on features.  
  Insight Reference: Insight 1  

- **Enhanced User Experience:** Recent user experience enhancements in Notion highlight their commitment to intuitive design, reinforcing the importance of ease of use; we can leverage this insight by emphasizing our advanced user-friendly interfaces that prioritize seamless integration and user satisfaction.  
  Insight Reference: Insight 4  

- **Advanced Automation Features:** Notion's focus on automation capabilities is a strength, but there's an opportunity for us to differentiate by promoting our unique data visualization tools that n

In [36]:
# ============================================================
# TOKEN / COST INFORMATION
# ============================================================

print("\n")
print("=" * 70)
print("HIERARCHICAL TOKEN / COST INFORMATION")
print("=" * 70)

usage_metrics = getattr(result_hier, "token_usage", None)

if usage_metrics is not None:
    print(usage_metrics)
else:
    print("Token usage object was not returned by this CrewAI version.")
    print("Use the provider/OpenRouter usage information if available.")



HIERARCHICAL TOKEN / COST INFORMATION
total_tokens=90335 prompt_tokens=79967 cached_prompt_tokens=20480 completion_tokens=10368 reasoning_tokens=0 cache_creation_tokens=0 successful_requests=32


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 420e8b43-682b-4845-8cf2-75caff228cd3                                                                       │
│  Final Output: **Headline:** Elevate Your Team's Productivity with Tailored Solutions                           │
│                                                                                                                 │
│  - **Compelling Pricing Strategy:** Notion's flexible pricing tiers cater to diverse business needs, making it  │
│  accessible for teams of all sizes, which presents an opportunity for us to position our offerings as a         │
│  cost-effective alternative for enterprises seeking tailored solutions without compromising on features.        │
│    Insight Reference: Insight 1                                                                                 │
│                                                                                                                 │
│  - **Enhanced User Experience:** Recent user experience enhancements in Notion highlight their commitment to    │
│  intuitive design, reinforcing the importance of ease of use; we can leverage this insight by emphasizing our   │
│  advanced user-friendly interfaces that prioritize seamless integration and user satisfaction.                  │
│    Insight Reference: Insight 4                                                                                 │
│                                                                                                                 │
│  - **Advanced Automation Features:** Notion's focus on automation capabilities is a strength, but there's an    │
│  opportunity for us to differentiate by promoting our unique data visualization tools that not only automate    │
│  processes but also provide actionable insights, empowering teams to make data-driven decisions faster.         │
│    Insight Reference: Insight 5                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [37]:
# ============================================================
# FINAL OUTPUT
# ============================================================

final_hier_output = str(result_hier)

print("=" * 70)
print("FINAL HIERARCHICAL RESULT")
print("=" * 70)
print(final_hier_output)

FINAL HIERARCHICAL RESULT
**Headline:** Elevate Your Team's Productivity with Tailored Solutions  

- **Compelling Pricing Strategy:** Notion's flexible pricing tiers cater to diverse business needs, making it accessible for teams of all sizes, which presents an opportunity for us to position our offerings as a cost-effective alternative for enterprises seeking tailored solutions without compromising on features.  
  Insight Reference: Insight 1  

- **Enhanced User Experience:** Recent user experience enhancements in Notion highlight their commitment to intuitive design, reinforcing the importance of ease of use; we can leverage this insight by emphasizing our advanced user-friendly interfaces that prioritize seamless integration and user satisfaction.  
  Insight Reference: Insight 4  

- **Advanced Automation Features:** Notion's focus on automation capabilities is a strength, but there's an opportunity for us to differentiate by promoting our unique data visualization tools that no

In [38]:
# ============================================================
# SEQUENTIAL BASELINE
# ============================================================

start_time = time.perf_counter()

result_seq = await sequential_crew.kickoff_async(
    inputs={
        "competitor": "Notion"
    }
)

elapsed_seq = time.perf_counter() - start_time

print("\n")
print("=" * 70)
print("SEQUENTIAL CREW OUTPUT")
print("=" * 70)
print(result_seq)

print("\n")
print("=" * 70)
print("SEQUENTIAL EXECUTION TIME")
print("=" * 70)
print(f"{elapsed_seq:.2f} seconds")

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 09c83cc9-1f22-4595-af25-f76d37bd9ce3                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│  Research Notion's products, current pricing,                                                                   │
│  and recent marketing/announcement activity.                                                                    │
│                                                                                                                 │
│  CURRENT DATE:                                                                                                  │
│  2026-08-14                                                                                                     │
│                                                                                                                 │
│  RECENT-ACTIVITY CUTOFF:                                                                                        │
│  2026-02-15                                                                                                     │
│                                                                                                                 │
│  ============================================================                                                   │
│  RESEARCH SCOPE                                                                                                 │
│  ============================================================                                                   │
│                                                                                                                 │
│  A. CURRENT PRODUCTS                                                                                            │
│                                                                                                                 │
│  Research Notion's currently offered                                                                            │
│  products/platform offerings.                                                                                   │
│                                                                                                                 │
│  B. CURRENT PRICING                                                                                             │
│                                                                                                                 │
│  Research Notion's current pricing tiers/plans.                                                                 │
│                                                                                                                 │
│  C. RECENT MARKETING CAMPAIGNS                                                                                  │
│                                                                                                                 │
│  Find marketing campaigns published between                                                                     │
│  2026-02-15 and 2026-08-14.                                                                                     │
│                                                                                                                 │
│  D. RECENT ANNOUNCEMENTS / PRODUCT ACTIVITY                                                                     │
│                                                                                                                 │
│  Find product launches, major announcements,                                                                    │
│  updates, or other official activity published         

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Market Research Analyst                                                                                 │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Research Notion's products, current pricing,                                                                   │
│  and recent marketing/announcement activity.                                                                    │
│                                                                                                                 │
│  CURRENT DATE:                                                                                                  │
│  2026-08-14                                                                                                     │
│                                                                                                                 │
│  RECENT-ACTIVITY CUTOFF:                                                                                        │
│  2026-02-15                                                                                                     │
│                                                                                                                 │
│  ============================================================                                                   │
│  RESEARCH SCOPE                                                                                                 │
│  ============================================================                                                   │
│                                                                                                                 │
│  A. CURRENT PRODUCTS                                                                                            │
│                                                                                                                 │
│  Research Notion's currently offered                                                                            │
│  products/platform offerings.                                                                                   │
│                                                                                                                 │
│  B. CURRENT PRICING                                                                                             │
│                                                                                                                 │
│  Research Notion's current pricing tiers/plans.                                                                 │
│                                                                                                                 │
│  C. RECENT MARKETING CAMPAIGNS                                                                                  │
│                                                                                                                 │
│  Find marketing campaigns published between                                                                     │
│  2026-02-15 and 2026-08-14.                                                                                     │
│                                                                                                                 │
│  D. RECENT ANNOUNCEMENTS / PRODUCT ACTIVITY                                                                     │
│                                                                                                                 │
│  Find product launches, major announcements,           

╭──────────────────────────────────────── 🔧 Tool Execution Started (#25) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: tavily_search                                                                                            │
│  Args: {'query': 'Notion current products site:notion.so'}                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#26) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: tavily_search                                                                                            │
│  Args: {'query': 'Notion current pricing site:notion.so'}                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Marketing Content Strategist                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Headline: Leverage Notion's Comprehensive Offerings for Diverse Needs                                          │
│                                                                                                                 │
│  Bullet 1: Highlight the competitive advantage of diverse product offerings | Insight: 1                        │
│                                                                                                                 │
│  Bullet 2: Emphasize the appeal of expanded plans for broader audience access | Insight: 2                      │
│                                                                                                                 │
│  Bullet 3: Showcase commitment to user experience through efficiency updates | Insight: 3                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Review ONLY the analyst's insights.                                                                            │
│                                                                                                                 │
│  Create one concise stakeholder-ready marketing                                                                 │
│  positioning angle for our company.                                                                             │
│                                                                                                                 │
│  ============================================================                                                   │
│  IMPORTANT                                                                                                      │
│  ============================================================                                                   │
│                                                                                                                 │
│  You ONLY know what is contained in the analyst's insights.                                                     │
│                                                                                                                 │
│  DO NOT invent:                                                                                                 │
│                                                                                                                 │
│  - our company's prices                                                                                         │
│  - our company's features                                                                                       │
│  - our company's products                                                                                       │
│  - our company's capabilities                                                                                   │
│  - our company's customer base                                                                                  │
│  - our company's support model                                                                                  │
│  - our company's advantages                                                                                     │
│  - results or claims about our company                                                                          │
│                                                                                                                 │
│  You may phrase a positioning DIRECTION,                                                                        │
│  but do not state an unverified capability                                                                      │
│  as an existing fact.                                                                                           │
│                                                                                                                 │
│  Example:                                                                                                       │
│                                                                                                                 │
│  Instead of:                                                                                                    │
│                                                                                                                 │
│  "Our affordable pricing beats Notion."                



SEQUENTIAL CREW OUTPUT
Headline: Leverage Notion's Comprehensive Offerings for Diverse Needs

Bullet 1: Highlight the competitive advantage of diverse product offerings | Insight: 1

Bullet 2: Emphasize the appeal of expanded plans for broader audience access | Insight: 2

Bullet 3: Showcase commitment to user experience through efficiency updates | Insight: 3


SEQUENTIAL EXECUTION TIME
15.01 seconds


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [39]:
# ============================================================
# SEQUENTIAL TOKEN USAGE
# ============================================================

seq_usage = getattr(result_seq, "token_usage", None)

print("=" * 70)
print("SEQUENTIAL TOKEN / COST INFORMATION")
print("=" * 70)

if seq_usage is not None:
    print(seq_usage)
else:
    print("Token usage object was not returned by this CrewAI version.")

SEQUENTIAL TOKEN / COST INFORMATION
total_tokens=90628 prompt_tokens=82002 cached_prompt_tokens=17536 completion_tokens=8626 reasoning_tokens=0 cache_creation_tokens=0 successful_requests=30


## Sequential vs. Hierarchical Comparison

Both CrewAI configurations were evaluated on the same Notion competitor-research and marketing task across three runs.

### Quantitative Comparison

| Metric | Sequential CrewAI | Hierarchical CrewAI |
|---|---:|---:|
| Average total tokens | **64,485** | **61,431** |
| Average requests | **21.3** | **23.0** |
| Average latency | **18.45 s** | **46.23 s** |
| Average manual quality | **12/15** | **9.3/15** |
| Observed reliability | Higher | Lower |

### Quality of Output

Based on manual evaluation across three runs, the Sequential crew achieved a higher average quality score (**12/15**) than the Hierarchical crew (**9.3/15**).

The Sequential crew generally produced clear and focused marketing outputs. However, some insight references did not consistently match the underlying research findings, showing that the handoff between agents could be improved.

The Hierarchical crew produced polished marketing-style outputs, but some runs contained claims that were not clearly supported by the available research. Therefore, the manager agent did not provide a measurable quality advantage for this particular task.

### Latency

Sequential execution was substantially faster.

- Sequential average latency: **18.45 seconds**
- Hierarchical average latency: **46.23 seconds**

The hierarchical approach was approximately **2.5× slower** than the sequential approach. This additional latency is likely due to the manager agent's coordination and delegation overhead.

### Token Usage / Cost

The average token usage was:

- Sequential: **64,485 tokens**
- Hierarchical: **61,431 tokens**

The Hierarchical crew used approximately **4.7% fewer total tokens** than the Sequential crew.

However, it generated more requests on average:

- Sequential: **21.3 requests**
- Hierarchical: **23.0 requests**

Therefore, lower total token usage did not translate into better overall efficiency because the Hierarchical crew had substantially higher latency and lower manual quality in these runs.

Since the exact dollar cost depends on the model's input/output pricing, token usage is used as the primary cost proxy.

### Reliability

For these three runs, the Sequential crew was more predictable because the workflow followed a fixed sequence:

1. Research
2. Analyze findings
3. Generate marketing output

This fixed structure made the workflow easier to understand and debug.

The Hierarchical crew introduced an additional manager layer. While this enables dynamic delegation, some outputs contained unsupported or weakly grounded claims. Therefore, the hierarchical approach showed lower observed reliability for this particular task.

### Pros, Cons, and When to Use

| Approach | Pros | Cons | When to Use |
|---|---|---|---|
| **Sequential** | Simple workflow; predictable execution order; easier to debug; lower latency | Fixed workflow; limited dynamic delegation | Use when tasks have a clear order and each agent depends on the previous agent's output |
| **Hierarchical** | Dynamic delegation; manager can coordinate and review agents; suitable for complex workflows | Higher latency; more coordination overhead; harder to debug; manager may introduce unnecessary decisions | Use when tasks are complex, require dynamic delegation, or need manager-level review |

### Conclusion

For this specific task, Sequential CrewAI performed better overall. It achieved a higher average manual quality score (12/15 vs. 9.3/15), substantially lower latency (18.45s vs. 46.23s), fewer average requests (21.33 vs. 23.00), and slightly lower estimated cost ($0.01242 vs. $0.01261), while using only slightly more tokens than the Hierarchical approach.

The Hierarchical approach remains useful for more complex problems where a manager needs to dynamically delegate work, review intermediate results, and coordinate multiple specialists. For this relatively well-defined competitor-research workflow, however, the additional management layer did not provide enough benefit to justify its higher latency and complexity. Therefore, Sequential CrewAI was the more appropriate architecture for this task.



# Task 5: Evaluation & Cost Awareness

## Objective

The goal of this task is to compare three approaches for the same Notion competitor-research problem:

1. Single-Agent LangGraph
2. CrewAI Sequential Multi-Agent Crew
3. CrewAI Hierarchical Multi-Agent Crew

The comparison focuses on output quality, token usage, latency, number of LLM/tool calls, reliability, and the additional complexity introduced by multi-agent orchestration.


In [40]:
def get_usage_metrics(crew):
    """
    Retrieve usage metrics recorded by CrewAI after the crew
    has successfully completed execution.
    """

    usage = getattr(crew, "usage_metrics", None)

    if usage is None:
        return None

    # Pydantic model
    if hasattr(usage, "model_dump"):
        return usage.model_dump()

    # Dictionary
    if isinstance(usage, dict):
        return usage

    # Object with attributes
    if hasattr(usage, "__dict__"):
        return vars(usage)

    return {"usage": usage}


def extract_metric(metrics, *names):
    """
    Try multiple possible metric names because the exact
    UsageMetrics structure can vary between CrewAI versions.
    """

    if metrics is None:
        return None

    for name in names:

        if isinstance(metrics, dict) and name in metrics:
            return metrics[name]

        if hasattr(metrics, name):
            return getattr(metrics, name)

    return None


def build_usage_record(name, crew):

    metrics = get_usage_metrics(crew)

    return {
        "system": name,

        "prompt_tokens": extract_metric(
            metrics,
            "prompt_tokens",
            "input_tokens"
        ),

        "completion_tokens": extract_metric(
            metrics,
            "completion_tokens",
            "output_tokens"
        ),

        "total_tokens": extract_metric(
            metrics,
            "total_tokens"
        ),
    }


# BUILD COMPARISON

usage_comparison = [
    build_usage_record(
        "Sequential (3 agents)",
        sequential_crew
    ),

    build_usage_record(
        "Hierarchical (3 agents + manager)",
        hierarchical_crew
    ),
]


# DISPLAY RESULTS

print("\n")
print("=" * 80)
print("CREWAI USAGE COMPARISON")
print("=" * 80)

for row in usage_comparison:

    print(
        f"{row['system']:40s} | "
        f"prompt={row['prompt_tokens']} | "
        f"completion={row['completion_tokens']} | "
        f"total={row['total_tokens']}"
    )



CREWAI USAGE COMPARISON
Sequential (3 agents)                    | prompt=82002 | completion=8626 | total=90628
Hierarchical (3 agents + manager)        | prompt=79967 | completion=10368 | total=90335


## 5.1 Success Criteria

Three manual success criteria were used to evaluate the generated outputs. Each criterion was scored from **1 to 5**, where 1 represents poor performance and 5 represents excellent performance.

### 1. Factual Grounding

The output should remain consistent with the available research findings and avoid unsupported claims, invented features, prices, or competitor information.

### 2. Completeness

The output should adequately address the requested task and provide the required marketing information based on the available evidence.

### 3. Marketing Relevance & Clarity

The output should be concise, understandable, professional, stakeholder-ready, and useful as a marketing positioning angle.

The maximum score for each run was therefore **15 points**.



# 5.2 Single-Agent LangGraph Results

The single-agent LangGraph solution was executed three times using the same Notion competitor-research task.

### Token, Calls and Latency

| Run | Prompt Tokens | Completion Tokens | Total Tokens | LLM Calls | Latency |
|---|---:|---:|---:|---:|---:|
| Run 1 | 3,129 | 510 | 3,639 | 5 | 15.39 s |
| Run 2 | 3,139 | 466 | 3,605 | 5 | 2.64 s |
| Run 3 | 3,145 | 457 | 3,602 | 5 | 13.45 s |
| **Average** | **3,138** | **477.7** | **3,615.3** | **5.0** | **10.49 s** |

### Manual Quality Evaluation

| Run | Factual Grounding /5 | Completeness /5 | Marketing Relevance & Clarity /5 | Total /15 |
|---|---:|---:|---:|---:|
| Run 1 | 4 | 3 | 4 | **11** |
| Run 2 | 4 | 3 | 4 | **11** |
| Run 3 | 4 | 3 | 4 | **11** |
| **Average** | **4.0** | **3.0** | **4.0** | **11.0/15** |

### Observation

The single-agent LangGraph solution was the most token-efficient approach and required only five LLM calls per run. Its main limitation was completeness: the generated answer repeatedly stated that the available research did not provide enough evidence to identify Notion's weaknesses. This was preferable to inventing unsupported information, but it reduced the completeness score.



# 5.3 CrewAI Sequential Results

The Sequential CrewAI implementation used three specialized agents:

1. Market Research Analyst
2. Business Insights Analyst
3. Marketing Content Strategist

The agents executed in a fixed sequence, with later tasks receiving the output of earlier tasks as context.

### Token, Requests and Latency

| Run | Prompt Tokens | Completion Tokens | Total Tokens | Requests | Latency |
|---|---:|---:|---:|---:|---:|
| Run 1 | 34,045 | 3,648 | 37,693 | 12 | 15.21 s |
| Run 2 | 59,093 | 6,040 | 65,133 | 22 | 25.13 s |
| Run 3 | 82,002 | 8,626 | 90,628 | 30 | 15.01 s |
| **Average** | **58,380** | **6,104.7** | **64,484.7** | **21.3** | **18.45 s** |

### Manual Quality Evaluation

| Run | Factual Grounding /5 | Completeness /5 | Marketing Relevance & Clarity /5 | Total /15 |
|---|---:|---:|---:|---:|
| Run 1 | 3 | 4 | 5 | **12** |
| Run 2 | 3 | 4 | 5 | **12** |
| Run 3 | 3 | 4 | 5 | **12** |
| **Average** | **3.0** | **4.0** | **5.0** | **12.0/15** |

### Observation

The Sequential crew produced concise and marketing-oriented outputs and achieved the highest average quality score among the three approaches.

However, some runs showed inconsistencies between the insight numbers referenced by the Marketing Content Strategist and the actual content of the corresponding insights. This demonstrated that downstream agents need explicit structured output contracts and reference rules.

The Sequential approach was considerably more efficient in latency than the Hierarchical approach, but it consumed substantially more tokens than the single-agent baseline.



# 5.4 CrewAI Hierarchical Results

The Hierarchical CrewAI implementation used the same specialized agents but added a manager agent responsible for coordinating and reviewing the work.

### Token, Requests and Latency

| Run | Prompt Tokens | Completion Tokens | Total Tokens | Requests | Latency |
|---|---:|---:|---:|---:|---:|
| Run 1 | 31,849 | 5,150 | 36,999 | 14 | 51.04 s |
| Run 2 | 49,822 | 7,137 | 56,959 | 23 | 42.33 s |
| Run 3 | 79,967 | 10,368 | 90,335 | 32 | 45.32 s |
| **Average** | **53,879.3** | **7,551.7** | **61,431.0** | **23.0** | **46.23 s** |

### Manual Quality Evaluation

| Run | Factual Grounding /5 | Completeness /5 | Marketing Relevance & Clarity /5 | Total /15 |
|---|---:|---:|---:|---:|
| Run 1 | 3 | 3 | 3 | **9** |
| Run 2 | 3 | 3 | 3 | **9** |
| Run 3 | 2 | 4 | 4 | **10** |
| **Average** | **2.7** | **3.3** | **3.3** | **9.3/15** |

### Observation

The Hierarchical crew introduced an additional manager layer for delegation and review. However, the measured outputs did not consistently demonstrate better factual grounding or overall quality.

Some hierarchical outputs introduced competitor comparisons or marketing claims that were not clearly supported by the available research. The manager therefore added coordination overhead without providing a clear quality advantage for this particular task.

The hierarchical approach also had substantially higher latency than both the single-agent and Sequential approaches.



# 5.5 Overall Comparison

| Metric | Single-Agent LangGraph | CrewAI Sequential | CrewAI Hierarchical |
|---|---:|---:|---:|
| **Average Total Tokens** | **3,615** | **64,485** | **61,431** |
| **Average LLM/Tool Requests** | **5.0** | **21.3** | **23.0** |
| **Average Latency** | **10.49 s** | **18.45 s** | **46.23 s** |
| **Average Quality** | **11.0/15** | **12.0/15** | **9.3/15** |
| Specialized Agents | No | Yes | Yes |
| Manager Agent | No | No | Yes |
| Coordination Overhead | Low | Medium | High |



# 5.6 Quality Comparison

The Sequential CrewAI approach achieved the highest manual quality score:

- **Sequential CrewAI:** 12.0/15
- **Single-Agent LangGraph:** 11.0/15
- **Hierarchical CrewAI:** 9.3/15

The Sequential crew performed particularly well in marketing relevance and clarity because the final agent was specifically responsible for converting the business insights into stakeholder-ready marketing content.

The Single-Agent solution was also strong and produced grounded answers, but it was less complete because the research evidence did not clearly establish Notion's weaknesses.

The Hierarchical crew produced usable marketing language, but the additional manager layer did not consistently improve factual grounding.



# 5.7 Token Usage and Cost Comparison

The average total token usage was:

- **Single-Agent LangGraph:** 3,615 tokens
- **CrewAI Hierarchical:** 61,431 tokens
- **CrewAI Sequential:** 64,485 tokens

Compared with the Single-Agent baseline:

- Sequential CrewAI used approximately **17.8×** as many tokens.
- Hierarchical CrewAI used approximately **17.0×** as many tokens.

Therefore, both multi-agent approaches introduced significant token overhead for this task.

Interestingly, the Hierarchical crew used slightly fewer tokens than the Sequential crew:

**61,431 vs. 64,485 tokens**

However, the Hierarchical approach was substantially slower and produced a lower average quality score.

Since the exact dollar cost depends on the LLM provider and model pricing, token usage is used as the primary cost proxy in this experiment.



# 5.8 Latency Comparison

The measured average latency was:

- **Single-Agent LangGraph:** 10.49 seconds
- **Sequential CrewAI:** 18.45 seconds
- **Hierarchical CrewAI:** 46.23 seconds

The Sequential crew was approximately **1.76× slower** than the Single-Agent solution.

The Hierarchical crew was approximately **4.41× slower** than the Single-Agent solution.

The Hierarchical crew was also approximately **2.51× slower** than the Sequential crew.

This demonstrates that the manager/delegation layer introduced substantial coordination overhead for this relatively well-defined workflow.



# 5.9 Reliability

Reliability was evaluated based on consistency, predictability, adherence to the task requirements, and correctness of information handoffs.

### Single-Agent LangGraph

The Single-Agent approach was relatively predictable and easy to debug because the complete workflow was controlled by one agent and a defined LangGraph state flow.

Its main limitation was that the final answer did not always provide complete competitor analysis because the available search results were insufficient to establish some claims.

### Sequential CrewAI

The Sequential crew was reliable in terms of workflow execution because the order was fixed:

1. Research
2. Business analysis
3. Marketing generation

However, the handoff between the Business Insights Analyst and Marketing Content Strategist required stronger formatting constraints. In some runs, insight references did not consistently match the corresponding insight content.

### Hierarchical CrewAI

The Hierarchical crew had more coordination complexity because a manager agent was responsible for delegation and review.

Although this can be useful for complex workflows, it introduced more variability and did not improve the measured quality for this particular task.



# 5.10 Output Handoff Issue and Fix

One important issue observed during the Sequential workflow was confusion between **Research Finding numbers** and **Insight numbers**.

Task 1 generated research findings such as:

```text
Finding 1
Finding 2
Finding 3

| Aspect                         | Sequential                                                                 | Hierarchical                                                             |
| ------------------------------ | -------------------------------------------------------------------------- | ------------------------------------------------------------------------ |
| **Pros**                       | Predictable workflow, simple coordination, lower latency, easier debugging | Dynamic delegation, manager-level review, suitable for complex workflows |
| **Cons**                       | Fixed execution order, limited dynamic delegation                          | Higher latency, more coordination overhead, more complex debugging       |
| **Quality in this experiment** | Higher                                                                     | Lower                                                                    |
| **Token Usage**                | High                                                                       | High                                                                     |
| **Latency**                    | Lower                                                                      | Much higher                                                              |
| **Reliability**                | More predictable                                                           | More variable                                                            |
| **Best Use Case**              | Clearly defined multi-step workflows                                       | Complex workflows requiring dynamic delegation and review                |


# 5.11 When to Use Each Architecture

### When to Use Sequential

Sequential execution is appropriate when:

* The workflow has a known order.
* Each stage depends on the previous stage.
* Tasks have clearly defined responsibilities.
* Predictability and lower latency are important.
* Easy debugging is required.

### When to Use Hierarchical

Hierarchical execution is more appropriate when:

* The workflow is complex.
* Tasks require dynamic delegation.
* A manager needs to review intermediate results.
* Different specialists may need to be selected dynamically.
* The task cannot be easily represented as a fixed sequence.



# 5.12 Single-Agent vs. Multi-Agent

The experiment shows that the Single-Agent LangGraph solution was substantially more efficient in terms of token usage and latency.

However, the Sequential CrewAI solution achieved the highest manual quality score:

* Single-Agent: **11.0/15**
* Sequential CrewAI: **12.0/15**
* Hierarchical CrewAI: **9.3/15**

This suggests that specialization can provide a quality benefit when the task naturally separates into research, analysis, and content-generation stages.

The Hierarchical crew did not provide a corresponding quality improvement in this experiment and introduced additional coordination overhead.



# 5.13 Cost Awareness

For this experiment, estimated monetary cost was calculated using the
reference token rates used by the notebook:

* Input: **$0.15 per 1M tokens**
* Output: **$0.60 per 1M tokens**

These are estimated costs rather than actual provider billing amounts;
actual charges may vary depending on the model and provider.

Based on the measured three-run averages:

| Architecture | Avg Total Tokens | Avg Estimated Cost |
|---|---:|---:|
| Single-Agent LangGraph | ~3,615 | ~$0.00076 |
| Sequential CrewAI | 64,484.67 | $0.012420 |
| Hierarchical CrewAI | 61,431.00 | $0.012613 |


Although Hierarchical CrewAI used fewer total tokens than Sequential
CrewAI, it generated more completion tokens, resulting in a slightly
higher estimated cost.

The multi-agent approaches consumed approximately **17–18× more tokens**
than the Single-Agent baseline.

Therefore, multi-agent orchestration should be introduced when the
benefits of specialization, delegation, or quality justify the additional
model usage, cost, and latency.



# 5.14 Was the Multi-Agent Crew Worth It?

For this specific Notion competitor-research task, the multi-agent
approach provided a small quality improvement when using the Sequential
configuration, increasing the average manual score from **11.0/15 to
12.0/15**.

However, this improvement came with a substantial increase in token
usage, from approximately **3,615 tokens to 64,485 tokens**, as well as
higher latency.

The Hierarchical configuration was not worth the additional complexity
for this task. It consumed approximately **61,431 tokens**, had an
average latency of **46.23 seconds**, and achieved a lower quality score
of **9.3/15**.

Therefore, for this particular task, a **well-designed Single-Agent
solution is the most cost-efficient choice**, while a **Sequential
multi-agent crew is justified when the additional specialization and
quality benefit are valuable enough to justify the additional cost and
latency**.

# 5.15 Final Conclusion

The experiment demonstrates that there is no universally best agent
architecture; the appropriate architecture depends on task complexity
and the desired trade-off between quality, cost, and latency.

For the Notion competitor-research workflow, the **Single-Agent
LangGraph** solution was the most efficient in terms of token usage,
latency, and estimated cost.

The **CrewAI Sequential** approach produced the highest manual quality
score (**12.0/15**) and benefited from clear role separation between
research, analysis, and marketing generation. However, it required
substantially more tokens and had a higher estimated cost than the
Single-Agent approach.

The **CrewAI Hierarchical** approach introduced dynamic management and
delegation but added significant latency and coordination overhead
without improving quality in these experiments. It also had a slightly
higher estimated cost than Sequential despite using fewer total tokens.

Overall, the results suggest that **Sequential CrewAI is the better
multi-agent architecture for this specific task, while Single-Agent
LangGraph is the best choice when minimizing cost, latency, and resource
usage is the primary objective**.

In [43]:
# ============================================================
# TASK 5 — 3-RUN TOKEN & COST ANALYSIS
# ============================================================

INPUT_PRICE_PER_1M = 0.15
OUTPUT_PRICE_PER_1M = 0.60


def calculate_cost(prompt_tokens, completion_tokens):
    input_cost = (prompt_tokens / 1_000_000) * INPUT_PRICE_PER_1M
    output_cost = (completion_tokens / 1_000_000) * OUTPUT_PRICE_PER_1M

    return input_cost + output_cost


def make_usage_record(
    system,
    run_number,
    prompt_tokens,
    completion_tokens,
    total_tokens,
    cached_prompt_tokens
):
    return {
        "System": system,
        "Run": run_number,
        "Prompt Tokens": prompt_tokens,
        "Completion Tokens": completion_tokens,
        "Total Tokens": total_tokens,
        "Cached Prompt Tokens": cached_prompt_tokens,
        "Estimated Cost ($)": calculate_cost(
            prompt_tokens,
            completion_tokens
        )
    }

In [44]:
# ============================================================
# ACTUAL 3-RUN DATA
# ============================================================

existing_runs = [

    # ---------------- SEQUENTIAL ----------------

    make_usage_record(
        "Sequential",
        1,
        34045,
        3648,
        37693,
        11136
    ),

    make_usage_record(
        "Sequential",
        2,
        59093,
        6040,
        65133,
        16256
    ),

    make_usage_record(
        "Sequential",
        3,
        82002,
        8626,
        90628,
        17536
    ),

    # ---------------- HIERARCHICAL ----------------

    make_usage_record(
        "Hierarchical",
        1,
        31849,
        5150,
        36999,
        3968
    ),

    make_usage_record(
        "Hierarchical",
        2,
        49822,
        7137,
        56959,
        15744
    ),

    make_usage_record(
        "Hierarchical",
        3,
        79967,
        10368,
        90335,
        20480
    )
]

In [45]:
import pandas as pd

runs_df = pd.DataFrame(existing_runs)

runs_df

,System,Run,Prompt Tokens,Completion Tokens,Total Tokens,Cached Prompt Tokens,Estimated Cost ($)
0,Sequential,1,34045,3648,37693,11136,0.007296
1,Sequential,2,59093,6040,65133,16256,0.012488
2,Sequential,3,82002,8626,90628,17536,0.017476
3,Hierarchical,1,31849,5150,36999,3968,0.007867
4,Hierarchical,2,49822,7137,56959,15744,0.011755
5,Hierarchical,3,79967,10368,90335,20480,0.018216


In [46]:
summary_df = (
    runs_df
    .groupby("System")
    .agg(
        Avg_Prompt_Tokens=("Prompt Tokens", "mean"),
        Avg_Completion_Tokens=("Completion Tokens", "mean"),
        Avg_Total_Tokens=("Total Tokens", "mean"),
        Avg_Cached_Tokens=("Cached Prompt Tokens", "mean"),
        Avg_Cost=("Estimated Cost ($)", "mean")
    )
    .reset_index()
)

summary_df

,System,Avg_Prompt_Tokens,Avg_Completion_Tokens,Avg_Total_Tokens,Avg_Cached_Tokens,Avg_Cost
0,Hierarchical,53879.333333,7551.666667,61431.000000,13397.333333,0.012613
1,Sequential,58380.000000,6104.666667,64484.666667,14976.000000,0.012420
